# OpenSetDGA — Re-run BiLSTM-OE (fixed OE lengths)

Chạy lại **bilstm_oe** sau khi fix bug truyền `lengths` cho OE batch:
- **Bug cũ:** `model(oe_x, return_feats=False)` → `lengths=None` → OE loss tính trên full padded sequence
- **Fix:** `model(oe_x, oe_lb, return_feats=False)` → OE batch dùng đúng actual lengths

**Seeds:** 1, 7, 42, 99, 314  
**Device:** GPU (T4 / P100)  
**Khuyến nghị:** Accelerator → GPU T4 x2 hoặc P100  
**Ước tính:** ~20–30 phút (5 seeds × ~4–5 phút/seed)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'huggingface_hub', 'tldextract'], check=True)
print('Packages ready.')

In [ ]:
import os
from pathlib import Path

WORKDIR = Path('/kaggle/working/OpenSetDGA-Detection')

if not WORKDIR.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/quanturong/OpenSetDGA-Detection.git',
         str(WORKDIR)],
        check=True
    )
    print('Repo cloned.')
else:
    subprocess.run(['git', '-C', str(WORKDIR), 'pull'], check=False)
    print('Repo already exists — pulled latest.')

os.chdir(WORKDIR)

# Verify OE lengths fix is present
src = (WORKDIR / 'src' / 'train_bilstm_oe.py').read_text(encoding='utf-8')
assert 'oe_lb = oe_batch[1].to(device)' in src, \
    'Fix not found: oe_lb missing — check git pull'
assert 'model(oe_x, oe_lb, return_feats=False)' in src, \
    'Fix not found: oe_lb not passed to model — check git pull'
print('Fix verified: oe_lb lengths passed correctly to model.')
print(f'Working dir: {os.getcwd()}')

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU found — training will be very slow on CPU.')

In [ ]:
from huggingface_hub import snapshot_download
import pandas as pd

DATA_DIR = WORKDIR / 'data' / 'processed'

if not (DATA_DIR / 'known' / 'train.csv').exists():
    print('Downloading dataset...')
    snapshot_download(
        repo_id='ThanhPhuongtphz/OpenSetDGA-Detection',
        repo_type='dataset',
        local_dir=str(DATA_DIR),
        ignore_patterns=['*.git*', '*.md', '*.txt'],
    )
    print('Download complete.')
else:
    print('Data already present.')

for p in [
    DATA_DIR / 'known' / 'train.csv',
    DATA_DIR / 'known' / 'test_known.csv',
    DATA_DIR / 'unknown_family' / 'test_unknown_family.csv',
    DATA_DIR / 'unknown_ood' / 'test_unknown_ood.csv',
]:
    n = len(pd.read_csv(p))
    print(f'  OK  {p.name}  ({n:,} rows)')

In [ ]:
SEEDS = [1, 7, 42, 99, 314]
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'=== Running bilstm_oe (5 seeds) on {DEVICE} ===')
result = subprocess.run(
    [sys.executable, 'src/run_multi_seed.py',
     '--seeds'] + [str(s) for s in SEEDS] + [
     '--only', 'bilstm_oe',
     '--device', DEVICE,
     '--run_dir', 'data/processed'],
    cwd=str(WORKDIR)
)
print(f'\nExit code: {result.returncode}')

In [ ]:
import json, numpy as np

SPLITS  = ['unknown_family', 'unknown_ood']
SCORERS = ['msp', 'energy']
MODEL   = 'bilstm_oe'

def load(seed):
    p = WORKDIR / 'baseline_out' / f'{MODEL}_s{seed}' / 'results.json'
    return json.load(open(p)) if p.exists() else {}

def stats(vals):
    vals = [v for v in vals if v is not None]
    if not vals: return None, None
    return np.mean(vals), np.std(vals)

print(f'BiLSTM-OE results (fixed OE lengths):')
print(f'{"Scorer/Split":<35} {"AUROC":>8} {"±":>6} {"AUPR":>8} {"FPR@95":>8} {"±":>6} {"Prec@95":>9}')
print('-' * 85)

for scorer in SCORERS:
    for split in SPLITS:
        key = f'ood_{scorer}_{split}'
        aurocs, auprs, fprs, precs = [], [], [], []
        missing = []
        for seed in SEEDS:
            d = load(seed).get(key, {})
            if d:
                aurocs.append(d['auroc'])
                auprs.append(d['aupr_out'])
                fprs.append(d['fpr_at_tpr'])
                precs.append(d.get('precision_at_tpr'))
            else:
                missing.append(seed)

        if missing:
            print(f'  MISSING seeds {missing} for {scorer}/{split}')
            continue

        ma, sa = stats(aurocs)
        mp, _  = stats(auprs)
        mf, sf = stats(fprs)
        mc, _  = stats(precs)
        label  = f'{scorer}/{split}'
        pc_str = f'{mc:.4f}' if mc is not None else '—'
        print(f'{label:<35} {ma:>8.4f} {sa:>6.4f} {mp:>8.4f} {mf:>8.4f} {sf:>6.4f} {pc_str:>9}')

print()
print('Per-seed AUPR detail:')
for scorer in SCORERS:
    for split in SPLITS:
        key = f'ood_{scorer}_{split}'
        vals = [load(s).get(key, {}).get('aupr_out') for s in SEEDS]
        v_str = '  '.join(f'{v:.4f}' if v is not None else '—' for v in vals)
        print(f'  {scorer}/{split}: {v_str}')

In [ ]:
import zipfile

zip_path = Path('/kaggle/working/bilstm_oe_fixed.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for seed in SEEDS:
        d = WORKDIR / 'baseline_out' / f'bilstm_oe_s{seed}'
        if not d.exists():
            print(f'WARNING: missing bilstm_oe_s{seed}')
            continue
        for f in d.rglob('*'):
            if f.is_file() and f.suffix in ('.json', '.pt', '.csv'):
                zf.write(f, f'bilstm_oe_s{seed}/{f.name}')

size_mb = zip_path.stat().st_size / (1024 ** 2)
print(f'Saved: {zip_path.name}  ({size_mb:.1f} MB)')
print('Download từ Output tab.')